<a href="https://colab.research.google.com/github/jiyeonlee-2930/Tourist-Prediction-Project-2026/blob/main/GW_Tourist_Forecast_Model(API%ED%98%B8%EC%B6%9C).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. 필요한 라이브러리
# ============================================================

import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from datetime import datetime
from google.colab import userdata

In [ ]:
# ============================================================
# 2. API 키 불러오기
# ============================================================

SERVICE_KEY = userdata.get("tourist-key")

print("API key loaded successfully.")

In [ ]:
# ============================================================
# 3. API URL 설정
# ============================================================

BASE_URL = "https://apis.data.go.kr/B551011/DataLabService/locgoRegnVisitrDDList"

In [ ]:
# ============================================================
# 4. API 호출 함수(한번만 사용)
# ============================================================

def request_api(start_date, end_date, page_no=1, num_rows=1000):

    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "pageNo": page_no,
        "MobileOS": "ETC",
        "MobileApp": "GangwonTourismResearch",
        "startYmd": start_date,
        "endYmd": end_date
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    return response.text

In [ ]:
# ============================================================
# 5. XML → DataFrame
# ============================================================

def parse_xml(xml_text):

    root = ET.fromstring(xml_text)

    rows = []

    for item in root.findall(".//item"):

        row = {
            "signguCode": item.findtext("signguCode"),
            "signguNm": item.findtext("signguNm"),
            "daywkDivCd": item.findtext("daywkDivCd"),
            "daywkDivNm": item.findtext("daywkDivNm"),
            "touDivCd": item.findtext("touDivCd"),
            "touDivNm": item.findtext("touDivNm"),
            "touNum": item.findtext("touNum"),
            "baseYmd": item.findtext("baseYmd")
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 6. 한 달 전체 데이터 수집 함수
# ============================================================

def collect_month(start_date, end_date):

    all_rows = []

    page_no = 1
    num_rows = 1000

    while True:

        print(
            f"Collecting {start_date} ~ {end_date}, "
            f"page {page_no}"
        )

        xml_text = request_api(
            start_date=start_date,
            end_date=end_date,
            page_no=page_no,
            num_rows=num_rows
        )

        root = ET.fromstring(xml_text)

        # totalCount 확인
        total_count_text = root.findtext(".//totalCount")

        if total_count_text is None:
            total_count = 0
        else:
            total_count = int(total_count_text)

        df_page = parse_xml(xml_text)

        if len(df_page) == 0:
            break

        all_rows.append(df_page)

        collected = page_no * num_rows

        if collected >= total_count:
            break

        page_no += 1

        # 서버 부담 방지
        time.sleep(0.3)

    if len(all_rows) == 0:
        return pd.DataFrame()

    return pd.concat(
        all_rows,
        ignore_index=True
    )

In [ ]:
##48개월 반복하여 월단위로 데이터 모집
# ============================================================
# 8. 월별 날짜 범위 생성
# ============================================================

periods = pd.period_range(
    start="2020-01",
    end="2023-12",
    freq="M"
)

periods[:5]

In [ ]:
# ============================================================
# 4. 강원특별자치도 18개 시군
# ============================================================

GANGWON_REGIONS = [
    "춘천시",
    "원주시",
    "강릉시",
    "동해시",
    "태백시",
    "속초시",
    "삼척시",
    "홍천군",
    "횡성군",
    "영월군",
    "평창군",
    "정선군",
    "철원군",
    "화천군",
    "양구군",
    "인제군",
    "고성군",
    "양양군"
]

In [ ]:
#4.
df_test, total_count = request_api(
    start_date="20230701",
    end_date="20251231",
    page_no=1,
    num_rows=300000
)

print("totalCount:", total_count)
print("received:", len(df_test))

In [ ]:
df_test, total_count = request_api(
    start_date="20230701",
    end_date="20251231",
    page_no=1,
    num_rows=10000
)

print("totalCount:", total_count)
print("received:", len(df_test))

In [ ]:
import time

# ============================================================
# 6. 하루 데이터 테스트
# ============================================================

xml_text = request_api(
    start_date="20210513",
    end_date="20210513",
    page_no=1,
    num_rows=1000
)

time.sleep(1) # Add a 1-second delay to avoid 'Too Many Requests' error

print(xml_text[:3000])

In [ ]:
df_test = parse_xml(xml_text)

display(df_test.head(20))

In [ ]:
df_gangwon_test = df_test[
    df_test["signguNm"].isin(GANGWON_REGIONS)
].copy()

display(df_gangwon_test)

In [ ]:
# ============================================================
# 10. 2020~2023 전체 수집
# ============================================================

all_months = []

for period in periods:

    start_date = period.start_time.strftime("%Y%m%d")
    end_date = period.end_time.strftime("%Y%m%d")

    print("=" * 60)
    print(period)
    print("=" * 60)

    try:

        df_month = collect_month(
            start_date=start_date,
            end_date=end_date
        )

        if len(df_month) > 0:

            # 강원 18개 시군만 선택
            df_month = df_month[
                df_month["signguNm"].isin(GANGWON_REGIONS)
            ].copy()

            all_months.append(df_month)

            print(
                "Gangwon records:",
                len(df_month)
            )

    except Exception as e:

        print(
            "ERROR:",
            period,
            e
        )

    time.sleep(0.5)

In [ ]:
# ============================================================
# 11. 전체 데이터 통합
# ============================================================

df_raw = pd.concat(
    all_months,
    ignore_index=True
)

print(df_raw.shape)

display(df_raw.head())

In [ ]:
# ============================================================
# 12. 데이터 타입 정리(날짜와 방문자 수를 숫자형으로 변환)
# ============================================================

df_raw["date"] = pd.to_datetime(
    df_raw["baseYmd"],
    format="%Y%m%d"
)

df_raw["touNum"] = pd.to_numeric(
    df_raw["touNum"],
    errors="coerce"
)

df_raw = df_raw.sort_values(
    ["date", "signguNm", "touDivCd"]
).reset_index(drop=True)

display(df_raw.head(20))

In [ ]:
# ============================================================
# 13. 원본 데이터 저장
# ============================================================

RAW_FILE = "Gangwon_daily_visitors_raw_2020_2023.csv"

df_raw.to_csv(
    RAW_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(RAW_FILE)

In [ ]:
##정선군/현지인(a), 정선군/외지인(b), 정선군/외국인(c)을 한 행으로 통합
# ============================================================
# 14. Long → Wide 변환
# ============================================================

df_wide = df_raw.pivot_table(
    index=[
        "date",
        "signguCode",
        "signguNm",
        "daywkDivNm"
    ],
    columns="touDivNm",
    values="touNum",
    aggfunc="first"
).reset_index()

df_wide.columns.name = None

display(df_wide.head())

In [ ]:
# ============================================================
# 15. 변수명 정리
# ============================================================

rename_dict = {
    "현지인(a)": "local_visitors",
    "외지인(b)": "nonlocal_visitors",
    "외국인(c)": "foreign_visitors"
}

df_wide = df_wide.rename(
    columns=rename_dict
)

display(df_wide.head())

In [ ]:
# ============================================================
# 16. 연구용 데이터 저장
# ============================================================

FINAL_FILE = "Gangwon_daily_visitors_2020_2023.csv"

df_wide.to_csv(
    FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    FINAL_FILE
)

# 데이터 수집 검증

In [ ]:
# ============================================================
# 17. 지역 수 확인
# ============================================================

regions = sorted(
    df_wide["signguNm"]
    .dropna()
    .unique()
)

print("Number of regions:", len(regions))

print(regions)

In [ ]:
# ============================================================
# 18. 날짜 범위 확인
# ============================================================
print(
    df_wide["date"].min()
)

print(
    df_wide["date"].max()
)


In [ ]:
# ============================================================
# 19. 지역별 데이터 개수 확인
# ============================================================
region_counts = (
    df_wide
    .groupby("signguNm")
    .size()
    .sort_values()
)

display(region_counts)

In [ ]:
# ============================================================
# 20. 결측치 확인
# ============================================================

missing = df_wide[
    [
        "local_visitors",
        "nonlocal_visitors",
        "foreign_visitors"
    ]
].isna().sum()

print(missing)

In [ ]:
##종속변수=외지인 방문자수
TARGET = "nonlocal_visitors"

In [ ]:
##API호출 및 데이터 콜렉션(한번만 사용)
def collect_day_all_pages(date):

    all_pages = []
    page_no = 1
    num_rows = 1000

    while True:
        xml_text = request_api(
            start_date=date,
            end_date=date,
            page_no=page_no,
            num_rows=num_rows
        )

        root = ET.fromstring(xml_text)

        total_count_text = root.findtext(".//totalCount")
        total_count = int(total_count_text) if total_count_text else 0

        df_page = parse_xml(xml_text)

        if df_page.empty:
            break

        all_pages.append(df_page)

        print(
            f"page={page_no}, "
            f"rows={len(df_page)}, "
            f"totalCount={total_count}"
        )

        if page_no * num_rows >= total_count:
            break

        page_no += 1
        time.sleep(0.2)

    return pd.concat(all_pages, ignore_index=True)

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import pandas as pd

df = pd.read_csv("/content/Gangwon_daily_visitors_2020_2023.csv")

print("Total rows:", len(df))
print("Number of regions:", df["signguNm"].nunique())
display(df.head())

① 받은 월은 다시 호출하지 않기
② API 호출 횟수를 직접 세기
③ 하루 900회 정도에서 자동 중단하기

In [ ]:
##API호출 검점하면서 데이터 수집
import math
import time

API_CALL_COUNT = 0
MAX_DAILY_CALLS = 900

def collect_month(start_date, end_date):

    global API_CALL_COUNT

    all_rows = []

    page_no = 1
    num_rows = 1000

    while True:

        # 일일 한도 보호
        if API_CALL_COUNT >= MAX_DAILY_CALLS:
            raise RuntimeError(
                f"안전을 위해 API 호출을 중단합니다. "
                f"현재 호출 수: {API_CALL_COUNT}"
            )

        xml_text = request_api(
            start_date=start_date,
            end_date=end_date,
            page_no=page_no,
            num_rows=num_rows
        )

        API_CALL_COUNT += 1

        print(
            f"API call {API_CALL_COUNT} | "
            f"{start_date}~{end_date} | "
            f"page {page_no}"
        )

        root = ET.fromstring(xml_text)

        total_count_text = root.findtext(".//totalCount")
        total_count = (
            int(total_count_text)
            if total_count_text
            else 0
        )

        df_page = parse_xml(xml_text)

        if df_page.empty:
            break

        all_rows.append(df_page)

        # 필요한 총 페이지 수
        total_pages = math.ceil(
            total_count / num_rows
        )

        if page_no >= total_pages:
            break

        page_no += 1
        time.sleep(0.5)

    if not all_rows:
        return pd.DataFrame()

    return pd.concat(
        all_rows,
        ignore_index=True
    )

In [ ]:
df["date"] = pd.to_datetime(df["date"])

print("지역 수:", df["signguNm"].nunique())
print("시작일:", df["date"].min())
print("종료일:", df["date"].max())

print("\n지역별 데이터 수")
display(df.groupby("signguNm").size().sort_values())

print("\n결측치")
print(df.isnull().sum())

print("\n지역-날짜 중복")
print(df.duplicated(subset=["date", "signguNm"]).sum())

In [ ]:
# 중복된 지역-날짜 확인
dup = df[df.duplicated(
    subset=["date", "signguNm"],
    keep=False
)].sort_values(["signguNm", "date"])

print("중복 행 수:", len(dup))

display(
    dup[[
        "date",
        "signguCode",
        "signguNm",
        "local_visitors",
        "nonlocal_visitors",
        "foreign_visitors"
    ]].head(30)
)

In [ ]:
dup_check = (
    df.groupby(["date", "signguNm"])
      .agg(
          n=("signguNm", "size"),
          local_nunique=("local_visitors", "nunique"),
          nonlocal_nunique=("nonlocal_visitors", "nunique"),
          foreign_nunique=("foreign_visitors", "nunique")
      )
      .query("n > 1")
)

print("중복 지역-날짜 조합 수:", len(dup_check))

different = (
    (dup_check["local_nunique"] > 1) |
    (dup_check["nonlocal_nunique"] > 1) |
    (dup_check["foreign_nunique"] > 1)
).sum()

print("값이 서로 다른 중복:", different)

display(dup_check.head(20))

In [ ]:
print("지역 수:", df["signguNm"].nunique())
print("중복:",
      df.duplicated(["date", "signguNm"]).sum())

In [ ]:
gosung = df[df["signguNm"] == "고성군"]

print(gosung["signguCode"].value_counts())

display(
    gosung[
        ["date", "signguCode", "signguNm",
         "local_visitors",
         "nonlocal_visitors",
         "foreign_visitors"]
    ].head(20)
)

In [ ]:
df["signguCode"] = df["signguCode"].astype(str)

df = df[
    df["signguCode"].str.startswith("51")
].copy()

print("지역 수:", df["signguNm"].nunique())
print("전체 행 수:", len(df))
print("중복:", df.duplicated(["date", "signguNm"]).sum())

In [ ]:
print("시작일:", df["date"].min())
print("종료일:", df["date"].max())

region_counts = (
    df.groupby("signguNm")
      .size()
      .sort_values()
)

display(region_counts)

In [ ]:
daily_region_count = (
    df.groupby("date")["signguNm"]
      .nunique()
)

print("날짜별 최소 지역 수:", daily_region_count.min())
print("날짜별 최대 지역 수:", daily_region_count.max())

display(
    daily_region_count.value_counts()
    .sort_index()
)

In [ ]:
import pandas as pd

# 0. 중복이 고성군에만 있는지, 여러 지역에 퍼져 있는지
dup_mask = df.duplicated(subset=['date','signguNm'], keep=False)
print(df[dup_mask]['signguNm'].value_counts())

# 1. (A) 검증 — 코드 컬럼이 살아있는지, 살아있다면 값이 몇 종류인지
for c in ['areaCd','areaNm','signguCd']:
    if c in df.columns:
        print(c, df.loc[dup_mask, c].unique()[:10])

# 2. (B) 검증 — 축 컬럼 잔존 여부
axis_cols = [c for c in df.columns
             if c in ['daywkDivCd','daywkDivNm','sexistseNm','ageGrp','touDivCd','touDivNm']]
print('잔존 축:', axis_cols)

# 3. (C) 검증 — 중복쌍의 값 차이 분포
g = df[dup_mask].groupby(['date','signguNm'])['local_visitors'].agg(['min','max'])
ratio = (g['max'] - g['min']) / g['max'].replace(0, pd.NA)
print(ratio.describe())

In [ ]:
gs = df[df['signguNm']=='고성군'].sort_values(['date','daywkDivNm'])

# 핵심: 중복쌍의 daywkDivNm이 서로 다른가?
print(gs['daywkDivNm'].value_counts(dropna=False))
print(gs.groupby('date')['daywkDivNm'].nunique().value_counts())

# 실제 두 행을 눈으로 확인
print(gs.head(6).to_string())

# 남아있는 전체 컬럼 (코드 컬럼이 정말 없는지)
print(df.columns.tolist())

# 고성군 날짜 범위 vs 다른 지역
print(gs['date'].min(), gs['date'].max(), gs['date'].nunique())
print(df.groupby('signguNm')['date'].nunique())

In [ ]:
from google.colab import userdata

KMA_API_KEY = userdata.get("KMA_API_KEY")

print("기상청 API Key loaded:", bool(KMA_API_KEY))

In [ ]:
import requests
import pandas as pd
from google.colab import userdata
from io import StringIO

KMA_API_KEY = userdata.get("KMA_API_KEY")

# authKey를 URL 쿼리 파라미터에서 제거
url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"

params = {
    "tm": "202301010900", # 날짜-시간 파라미터는 여기에서 지정
    "stn": "",
    "inf": "SFC",
    "help": "0",
    "authKey": KMA_API_KEY
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

print(response.text[:3000])

In [ ]:
import requests
from google.colab import userdata

KMA_API_KEY = userdata.get("KMA_API_KEY")

url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"
params = {
    "tm1": "20200101",
    "tm2": "20200110",
    "stn": "101",          # 춘천
    "disp": "0",
    "help": "1",
    "authKey": KMA_API_KEY
}

response = requests.get(url, params=params, timeout=30)

print("HTTP status:", response.status_code)
print(response.text[:5000])

In [ ]:
import requests
from google.colab import userdata

KMA_API_KEY = userdata.get("KMA_API_KEY")

url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"
params = {
    "tm1": "20200101",
    "tm2": "20200110",
    "stn": "101",      # 춘천
    "disp": "0",
    "help": "0",       # 실제 데이터 출력
    "authKey": KMA_API_KEY
}

response = requests.get(url, params=params, timeout=30)

print("HTTP status:", response.status_code)
print(response.text[:5000])

In [ ]:
import requests
import pandas as pd
from google.colab import userdata
from io import StringIO

KMA_API_KEY = userdata.get("KMA_API_KEY")

url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"

params = {
    "tm1": "20200101",
    "tm2": "20230630",
    "stn": "101",      # 춘천
    "disp": "1",       # CSV 형태
    "help": "0",
    "authKey": KMA_API_KEY
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

# 원본 그대로 먼저 저장
raw_path = "/content/KMA_ASOS_101_Chuncheon_20200101_20230630.txt"

with open(raw_path, "w", encoding="utf-8") as f:
    f.write(response.text)

print("저장 완료:", raw_path)
print("응답 크기:", len(response.text), "characters")

In [ ]:
with open(
    "/content/KMA_ASOS_101_Chuncheon_20200101_20230630.txt",
    "r",
    encoding="utf-8"
) as f:
    lines = f.readlines()

data_lines = [
    line.strip()
    for line in lines
    if line.strip()
    and not line.startswith("#")
]

print("데이터 행 수:", len(data_lines))

if len(data_lines) > 0:
    print("첫 행:", data_lines[0][:200])
    print("마지막 행:", data_lines[-1][:200])

In [ ]:
import requests
from google.colab import userdata

KMA_API_KEY = userdata.get("KMA_API_KEY")

url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"
params = {
    "tm1": "20200101",
    "tm2": "20230630",
    "stn": "101",       # 춘천
    "help": "0",
    "mode": "0",
    "authKey": KMA_API_KEY
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()

# 원본을 즉시 저장
path = "/content/KMA_ASOS_101_Chuncheon_20200101_20230630.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write(response.text)

print("저장 완료:", path)
print("응답 크기:", len(response.text))

In [ ]:
path = "/content/KMA_ASOS_101_Chuncheon_20200101_20230630.txt"

with open(path, "r", encoding="utf-8") as f:
    text = f.read()

print(text)

In [ ]:
import requests
from google.colab import userdata

KMA_API_KEY = userdata.get("KMA_API_KEY")

url = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"

params = {
    "tm1": "20200101",
    "tm2": "20230630",
    "stn": "101",          # 춘천
    "help": "0",
    "mode": "0",
    "authKey": KMA_API_KEY
}

r = requests.get(url, params=params, timeout=120)

print("HTTP:", r.status_code)

# 정상 응답일 때만 저장
if r.status_code == 200:
    path = "/content/KMA_ASOS_101_Chuncheon_20200101_20230630.txt"

    with open(path, "w", encoding="utf-8") as f:
        f.write(r.text)

    print("저장 완료:", path)
    print("파일 크기:", len(r.text))

else:
    print("다운로드 실패")
    print(r.text[:500])

In [ ]:
import os
import time
import calendar
import requests
import pandas as pd

from google.colab import userdata
from datetime import datetime

# =========================================================
# 1. 기본 설정
# =========================================================

KMA_API_KEY = userdata.get("KMA_API_KEY")

BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php"

START_DATE = "2020-01-01"
END_DATE   = "2023-06-30"

SAVE_DIR = "/content/kma_asos_hourly"
os.makedirs(SAVE_DIR, exist_ok=True)

# 기상청 공식 자료에서 확인되는 강원 주요 ASOS 지점
STATIONS = {
    90:  "Sokcho",
    95:  "Cheorwon",
    100: "Daegwallyeong",
    101: "Chuncheon",
    105: "Gangneung",
    114: "Wonju",
    211: "Inje",
    212: "Hongcheon",
    216: "Taebaek"
}

print("ASOS stations:", STATIONS)
print("저장 폴더:", SAVE_DIR)

In [ ]:
# =========================================================
# 2. 월별 기간 생성 함수
# =========================================================

def monthly_periods(start_date, end_date):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    periods = []

    current = pd.Timestamp(
        year=start.year,
        month=start.month,
        day=1
    )

    while current <= end:

        last_day = calendar.monthrange(
            current.year,
            current.month
        )[1]

        month_start = max(
            current,
            start
        )

        month_end = min(
            pd.Timestamp(
                year=current.year,
                month=current.month,
                day=last_day
            ),
            end
        )

        periods.append((month_start, month_end))

        current = current + pd.offsets.MonthBegin(1)

    return periods

In [ ]:
# =========================================================
# 3. 실제 ASOS 시간자료 다운로드
# =========================================================

def download_asos_month(stn, station_name, start_date, end_date):

    ym = start_date.strftime("%Y%m")

    filename = (
        f"ASOS_{stn}_{station_name}_{ym}.txt"
    )

    filepath = os.path.join(
        SAVE_DIR,
        filename
    )

    # 이미 다운로드한 파일은 다시 호출하지 않음
    if os.path.exists(filepath):

        if os.path.getsize(filepath) > 1000:
            print("SKIP:", filename)
            return True

    # 시간자료 기간조회 형식
    tm1 = start_date.strftime("%Y%m%d0000")
    tm2 = end_date.strftime("%Y%m%d2300")

    params = {
        "tm1": tm1,
        "tm2": tm2,
        "stn": str(stn),
        "help": "0",
        "authKey": KMA_API_KEY
    }

    try:

        r = requests.get(
            BASE_URL,
            params=params,
            timeout=120
        )

        if r.status_code != 200:
            print(
                "ERROR",
                stn,
                ym,
                "HTTP:",
                r.status_code
            )
            return False

        text = r.text

        # 너무 작은 응답은 오류 가능성이 있으므로 저장하지 않음
        if len(text) < 1000:

            print(
                "WARNING:",
                stn,
                ym,
                "응답 크기:",
                len(text)
            )

            print(text[:300])

            return False

        # 성공한 응답만 저장
        with open(
            filepath,
            "w",
            encoding="utf-8"
        ) as f:
            f.write(text)

        print(
            "SAVE:",
            filename,
            "size:",
            f"{len(text):,}"
        )

        return True

    except Exception as e:

        print(
            "ERROR:",
            stn,
            ym,
            e
        )

        return False

In [ ]:
# =========================================================
# 4. 기존 파일을 유지하면서 이어받기
# =========================================================

import os
import time
import pandas as pd

periods = monthly_periods(
    START_DATE,
    END_DATE
)

success_count = 0
skip_count = 0
failures = []

for stn, station_name in STATIONS.items():

    print("\n====================================")
    print(f"STN {stn} : {station_name}")
    print("====================================")

    for start_date, end_date in periods:

        ym = start_date.strftime("%Y%m")

        filename = f"ASOS_{stn}_{station_name}_{ym}.txt"
        filepath = os.path.join(SAVE_DIR, filename)

        # -------------------------------------------------
        # 이미 정상 저장된 파일은 API 호출하지 않음
        # -------------------------------------------------
        if os.path.exists(filepath) and os.path.getsize(filepath) > 1000:
            print("SKIP:", filename)
            skip_count += 1
            continue

        # -------------------------------------------------
        # 없는 파일만 실제 API 호출
        # -------------------------------------------------
        ok = download_asos_month(
            stn,
            station_name,
            start_date,
            end_date
        )

        if ok:
            success_count += 1

        else:
            print(
                f"FAIL → STN={stn}, "
                f"MONTH={ym}"
            )

            failures.append({
                "stn": stn,
                "station": station_name,
                "year_month": ym
            })

            # 실패해도 다음 월로 계속 진행
            # 같은 월을 즉시 재호출하지 않음

        # 호출 간격
        time.sleep(0.7)


print("\n====================================")
print("수집 작업 종료")
print("====================================")

print("기존 파일 SKIP :", skip_count)
print("이번에 성공     :", success_count)
print("실패             :", len(failures))

In [ ]:
# 실패 목록 저장
fail_df = pd.DataFrame(failures)

fail_path = "/content/KMA_ASOS_download_failures.csv"
fail_df.to_csv(
    fail_path,
    index=False,
    encoding="utf-8-sig"
)

print("실패 목록 저장:", fail_path)

if len(fail_df) > 0:
    display(fail_df)
else:
    print("모든 데이터 수집 성공")

In [ ]:
import os

SAVE_DIR = "/content/kma_asos_hourly"

files = sorted([
    f for f in os.listdir(SAVE_DIR)
    if f.endswith(".txt")
])

print("총 파일 수:", len(files))
print("첫 파일:", files[0])
print("마지막 파일:", files[-1])

In [ ]:
import os
import re
import pandas as pd
import numpy as np

SAVE_DIR = "/content/kma_asos_hourly"

files = sorted([
    os.path.join(SAVE_DIR, f)
    for f in os.listdir(SAVE_DIR)
    if f.endswith(".txt")
])

print("파일 수:", len(files))

In [ ]:
def read_kma_file(filepath):

    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # ----------------------------
    # 실제 데이터 행만 추출
    # ----------------------------
    data_lines = [
        line.strip()
        for line in lines
        if line.strip()
        and not line.startswith("#")
        and " " in line  # 쉼표(,) 대신 공백( )을 기준으로 필터링
    ]

    if len(data_lines) == 0:
        return None

    # 공백으로 분리된 값을 처리하도록 수정
    rows = [re.split(r"\s+", line) for line in data_lines]

    # 열 개수
    ncol = len(rows[0])

    # 파일 안에서 관측변수 이름이 있는 header 찾기
    header_candidates = []

    for line in lines:
        if line.startswith("#"):
            clean = line.lstrip("#").strip()
            # 'STN'과 'YYMMDD' 또는 'TM'을 포함하는 헤더 라인 찾기
            if "STN" in clean and ("YYMMDD" in clean or "TM" in clean):
                header_candidates.append(clean)

    # header 자동 인식 및 중복 컬럼명 처리
    columns = None

    for h in header_candidates:
        tokens = re.split(r"\s+", h)
        if len(tokens) == ncol:
            # 중복되는 컬럼 이름에 숫자 접미사를 추가하여 고유하게 만듦
            unique_columns = []
            counts = {}
            for col_name in tokens:
                base_name = col_name.strip()
                if base_name in counts:
                    counts[base_name] += 1
                    unique_columns.append(f"{base_name}_{counts[base_name]}")
                else:
                    counts[base_name] = 0 # 첫 번째 발생은 접미사 없이
                    unique_columns.append(base_name)
            columns = unique_columns
            break

    # header 자동 인식 실패 시 일단 col_번호 사용
    if columns is None:
        columns = [f"col_{i}" for i in range(ncol)]

    df_temp = pd.DataFrame(rows, columns=columns)

    # 원본 파일 정보도 보존
    df_temp["source_file"] = os.path.basename(filepath)

    return df_temp

In [ ]:
sample = read_kma_file(files[0])

if sample is not None:
    print("shape:", sample.shape)
    print()
    print("컬럼:")
    print(sample.columns.tolist())
    print()
    display(sample.head())
else:
    print(f"파일 {files[0]}에서 유효한 데이터를 찾을 수 없습니다.")

In [ ]:
hourly_list = []

for i, filepath in enumerate(files):

    temp = read_kma_file(filepath)

    if temp is not None:
        hourly_list.append(temp)

    if (i + 1) % 50 == 0:
        print(f"{i+1} / {len(files)} 파일 처리")

hourly = pd.concat(
    hourly_list,
    ignore_index=True
)

print("\n전체 시간자료 shape:", hourly.shape)
display(hourly.head())

In [ ]:
# 1. 컬럼명 앞뒤 공백 제거
sample.columns = sample.columns.str.strip()

print(sample.columns.tolist())

In [ ]:
import os
import pandas as pd

SAVE_DIR = "/content/kma_asos_hourly"

files = sorted([
    os.path.join(SAVE_DIR, f)
    for f in os.listdir(SAVE_DIR)
    if f.endswith(".txt")
])

hourly_list = []

for i, filepath in enumerate(files):

    temp = read_kma_file(filepath)

    if temp is not None:

        # 컬럼명 공백 제거
        # read_kma_file에서 이미 고유하게 처리했으므로
        # 여기서 다시 이름을 고유하게 만들 필요는 없지만, 앞뒤 공백 제거는 여전히 유효
        temp.columns = temp.columns.str.strip()

        hourly_list.append(temp)

    if (i + 1) % 50 == 0:
        print(f"{i+1}/{len(files)} 처리 완료")

hourly = pd.concat(
    hourly_list,
    ignore_index=True
)

print("\n전체 shape:", hourly.shape)
print("전체 파일:", len(files))
print("컬럼 수:", len(hourly.columns))

display(hourly.head())

In [ ]:
# 2. 시간 컬럼 변환

hourly["YYMMDDHHMI"] = (
    hourly["YYMMDDHHMI"]
    .astype(str)
    .str.strip()
)

hourly["datetime"] = pd.to_datetime(
    hourly["YYMMDDHHMI"],
    format="%Y%m%d%H%M",
    errors="coerce"
)

hourly["date"] = hourly["datetime"].dt.date

print("시작:", hourly["datetime"].min())
print("종료:", hourly["datetime"].max())
print("날짜 변환 실패:", hourly["datetime"].isna().sum())

In [ ]:
wanted = [
    "STN",
    "TA",
    "WS",
    "HM",
    "RN",
    "RN_DAY",
    "SD_DAY",
    "SD_TOT"
]

print("존재하는 변수:")

for c in wanted:
    print(c, ":", c in hourly.columns)

In [ ]:
numeric_cols = [
    "STN",
    "TA",
    "WS",
    "HM",
    "RN",
    "RN_DAY",
    "SD_DAY",
    "SD_TOT"
]

for c in numeric_cols:
    if c in hourly.columns:
        hourly[c] = pd.to_numeric(
            hourly[c],
            errors="coerce"
        )

In [ ]:
# 기상청 결측 코드 제거
weather_cols = [
    "TA",
    "WS",
    "HM",
    "RN",
    "RN_DAY",
    "SD_DAY",
    "SD_TOT"
]

for c in weather_cols:
    if c in hourly.columns:
        hourly.loc[
            hourly[c] <= -9,
            c
        ] = pd.NA

In [ ]:
daily = (
    hourly
    .groupby(
        ["STN", "date"],
        as_index=False
    )
    .agg(
        avg_temp=("TA", "mean"),
        max_temp=("TA", "max"),
        min_temp=("TA", "min"),

        avg_wind=("WS", "mean"),
        avg_humidity=("HM", "mean"),

        # RN_DAY는 누적값이므로 합계가 아닌 최대값
        rain=("RN_DAY", "max") if "RN_DAY" in hourly.columns else ("RN", "max"),

        new_snow=("SD_DAY", "max") if "SD_DAY" in hourly.columns else ("SD", "max"),
        snow_depth=("SD_TOT", "max") if "SD_TOT" in hourly.columns else ("SD_1", "max"),

        hourly_obs=("datetime", "count")
    )
)

In [ ]:
station_names = {
    90:  "Sokcho",
    95:  "Cheorwon",
    100: "Daegwallyeong",
    101: "Chuncheon",
    105: "Gangneung",
    114: "Wonju",
    211: "Inje",
    212: "Hongcheon",
    216: "Taebaek"
}

daily["station_name"] = daily["STN"].map(
    station_names
)

In [ ]:
print("관측소 수:", daily["STN"].nunique())
print("일자료 행 수:", len(daily))
print("시작일:", daily["date"].min())
print("종료일:", daily["date"].max())

print("\n관측소별 일수")
print(
    daily.groupby(
        ["STN", "station_name"]
    ).size()
)

In [ ]:
output_path = "/content/Gangwon_ASOS_daily_2020_202306.csv"

daily.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_path)

In [ ]:
# 1. 관측소-날짜 중복 확인
dup = (
    daily.groupby(["STN", "date"])
    .size()
    .reset_index(name="n")
)

dup = dup[dup["n"] > 1]

print("중복 관측소-날짜 수:", len(dup))
print("중복으로 추가된 행 수:", (dup["n"] - 1).sum())

display(dup.head(30))

In [ ]:
# 2. 관측소별 일수 확인
station_days = (
    daily.groupby(["STN", "station_name"])
    .agg(
        rows=("date", "size"),
        unique_days=("date", "nunique"),
        start_date=("date", "min"),
        end_date=("date", "max")
    )
    .reset_index()
)

display(station_days)

In [ ]:
# 3. 동일 관측소-날짜를 하나로 통합

daily_clean = (
    daily
    .groupby(
        ["STN", "station_name", "date"],
        as_index=False
    )
    .agg(
        avg_temp=("avg_temp", "mean"),
        max_temp=("max_temp", "max"),
        min_temp=("min_temp", "min"),
        rain=("rain", "max"),
        avg_wind=("avg_wind", "mean"),
        avg_humidity=("avg_humidity", "mean"),
        new_snow=("new_snow", "max"),
        snow_depth=("snow_depth", "max"),
        hourly_obs=("hourly_obs", "sum")
    )
)

In [ ]:
print("관측소 수:", daily_clean["STN"].nunique())
print("일자료 행 수:", len(daily_clean))
print("고유 날짜 수:", daily_clean["date"].nunique())
print("시작일:", daily_clean["date"].min())
print("종료일:", daily_clean["date"].max())

print(
    "\n관측소별 일수:\n",
    daily_clean.groupby(
        ["STN", "station_name"]
    )["date"].nunique()
)

In [ ]:
weather_cols = [
    "avg_temp",
    "max_temp",
    "min_temp",
    "rain",
    "avg_wind",
    "avg_humidity",
    "new_snow",
    "snow_depth"
]

print(daily_clean[weather_cols].isna().sum())

In [ ]:
# 현재 hourly에 들어 있는 실제 강수/적설 관련 컬럼 확인
rain_snow_cols = [
    c for c in hourly.columns
    if any(x in str(c).upper() for x in ["RN", "RAIN", "SD", "SNOW"])
]

print("강수/적설 관련 컬럼:")
for c in rain_snow_cols:
    print(repr(c))

In [ ]:
# 첫 번째 저장 파일의 원본 헤더를 그대로 확인
first_file = files[0]

with open(first_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines:
    if line.startswith("#"):
        print(line.rstrip())

In [ ]:
for i, c in enumerate(hourly.columns):
    print(f"{i:02d} : {repr(c)}")

In [ ]:
# 현재 pandas 컬럼 확인
print(hourly.columns.tolist())

rename_map = {
    "GST":   "GST_WD",
    "GST_1": "GST_WS",
    "GST_2": "GST_TM",

    "RN":    "RN",
    "RN_1":  "RN_DAY",
    "RN_2":  "RN_JUN",
    "RN_3":  "RN_INT",

    "SD":    "SD_HR3",
    "SD_1":  "SD_DAY",
    "SD_2":  "SD_TOT"
}

hourly = hourly.rename(columns=rename_map)

print([
    c for c in hourly.columns
    if c.startswith("RN") or c.startswith("SD")
])

In [ ]:
weather_cols = [
    "TA",
    "WS",
    "HM",
    "RN",
    "RN_DAY",
    "RN_INT",
    "SD_HR3",
    "SD_DAY",
    "SD_TOT"
]

for c in weather_cols:
    if c in hourly.columns:
        hourly[c] = pd.to_numeric(
            hourly[c],
            errors="coerce"
        )

In [ ]:
for c in weather_cols:
    if c in hourly.columns:
        hourly.loc[hourly[c] <= -9, c] = pd.NA

In [ ]:
daily = (
    hourly
    .groupby(["STN", "date"], as_index=False)
    .agg(
        avg_temp=("TA", "mean"),
        max_temp=("TA", "max"),
        min_temp=("TA", "min"),

        avg_wind=("WS", "mean"),
        avg_humidity=("HM", "mean"),

        # RN_DAY는 일 누적 강수량이므로 max 사용
        rain=("RN_DAY", "max"),

        # 적설
        new_snow=("SD_DAY", "max"),
        snow_depth=("SD_TOT", "max"),

        hourly_obs=("datetime", "count")
    )
)

In [ ]:
station_names = {
    90:  "Sokcho",
    95:  "Cheorwon",
    100: "Daegwallyeong",
    101: "Chuncheon",
    105: "Gangneung",
    114: "Wonju",
    211: "Inje",
    212: "Hongcheon",
    216: "Taebaek"
}

daily["station_name"] = daily["STN"].map(station_names)

In [ ]:
check_cols = [
    "avg_temp",
    "max_temp",
    "min_temp",
    "rain",
    "avg_wind",
    "avg_humidity",
    "new_snow",
    "snow_depth"
]

print(daily[check_cols].isna().sum())

In [ ]:
print("강수 > 0 일수:",
      (daily["rain"] > 0).sum())

print("신적설 > 0 일수:",
      (daily["new_snow"] > 0).sum())

print("적설 > 0 일수:",
      (daily["snow_depth"] > 0).sum())

print("\n강수 통계")
print(daily["rain"].describe())

print("\n신적설 통계")
print(daily["new_snow"].describe())

In [ ]:
# 시간별 RN도 숫자형/결측코드 처리
hourly["RN"] = pd.to_numeric(hourly["RN"], errors="coerce")
hourly.loc[hourly["RN"] <= -9, "RN"] = pd.NA

rain_verify = (
    hourly.groupby(["STN", "date"], as_index=False)
    .agg(
        rn_valid=("RN", "count"),
        rn_positive=("RN", lambda x: (x > 0).sum()),
        rn_max=("RN", "max"),
        rn_day_valid=("RN_DAY", "count"),
        rn_day_max=("RN_DAY", "max")
    )
)

# RN_DAY는 없지만 시간강수 RN은 양수인 날짜
problem_rain = rain_verify[
    (rain_verify["rn_day_valid"] == 0) &
    (rain_verify["rn_positive"] > 0)
]

print("RN_DAY는 NA인데 RN에서 비가 확인된 날:", len(problem_rain))
display(problem_rain.head(20))

In [ ]:
# 1. 강수량 NA → 0
daily["rain"] = daily["rain"].fillna(0)

# 2. 날짜형 변환 및 정렬
daily["date"] = pd.to_datetime(daily["date"])

daily = daily.sort_values(
    ["STN", "date"]
).reset_index(drop=True)

# 3. 온도 결측 31건 보간
temp_cols = [
    "avg_temp",
    "max_temp",
    "min_temp"
]

for col in temp_cols:
    daily[col] = (
        daily.groupby("STN")[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

# 4. new_snow는 전부 NA이므로 제외
daily = daily.drop(
    columns=["new_snow"],
    errors="ignore"
)

print(daily[
    [
        "avg_temp",
        "max_temp",
        "min_temp",
        "rain",
        "avg_wind",
        "avg_humidity",
        "snow_depth"
    ]
].isna().sum())

In [ ]:
weather_cols = [
    "avg_temp",
    "max_temp",
    "min_temp",
    "rain",
    "avg_wind",
    "avg_humidity",
    "snow_depth"
]

print("=== Missing values ===")
print(daily[weather_cols].isna().sum())

print("\nStations:", daily["STN"].nunique())
print("Rows:", len(daily))
print("Dates:", daily["date"].nunique())
print("Duplicates:",
      daily.duplicated(["STN", "date"]).sum())

In [ ]:
##정제본 저장
weather_path = "/content/Gangwon_ASOS_daily_final_2020_202306.csv"

daily.to_csv(
    weather_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", weather_path)

In [ ]:
import pandas as pd

visitor_path = "/content/Gangwon_daily_visitors_2020_2023.csv"

df = pd.read_csv(visitor_path)

print(df.shape)
print(df.columns.tolist())
display(df.head())

In [ ]:
region_station_map = {
    "춘천시": 101,
    "원주시": 114,
    "강릉시": 105,
    "동해시": 105,
    "태백시": 216,
    "속초시": 90,
    "삼척시": 216,
    "홍천군": 212,
    "횡성군": 114,
    "영월군": 216,
    "평창군": 100,
    "정선군": 216,
    "철원군": 95,
    "화천군": 101,
    "양구군": 211,
    "인제군": 211,
    "고성군": 90,
    "양양군": 90
}

df["station_id"] = df["signguNm"].map(region_station_map)

print(
    df[["signguNm", "station_id"]]
    .drop_duplicates()
    .sort_values("signguNm")
)

print("\n매핑 실패:",
      df["station_id"].isna().sum())

In [ ]:
#관광객 데이터 df와 기상 데이터 daily를 결합
# 날짜 형식 통일
df["date"] = pd.to_datetime(df["date"])
daily["date"] = pd.to_datetime(daily["date"])

# 기상자료의 STN → station_id
weather = daily.rename(columns={"STN": "station_id"}).copy()

df["station_id"] = df["station_id"].astype(int)
weather["station_id"] = weather["station_id"].astype(int)

# 필요한 기상변수
weather_cols = [
    "date",
    "station_id",
    "avg_temp",
    "max_temp",
    "min_temp",
    "rain",
    "avg_wind",
    "avg_humidity",
    "snow_depth"
]

# 관광객 + 기상자료 결합
df_model = df.merge(
    weather[weather_cols],
    on=["date", "station_id"],
    how="left",
    validate="many_to_one"
)

print("관광객 원본 행 수 :", len(df))
print("결합 후 행 수     :", len(df_model))

print("\n기상자료 결측치")
print(
    df_model[
        [
            "avg_temp",
            "max_temp",
            "min_temp",
            "rain",
            "avg_wind",
            "avg_humidity",
            "snow_depth"
        ]
    ].isna().sum()
)

# Clustering

In [ ]:
cluster_features = [
    "avg_temp",
    "rain",
    "avg_wind",
    "avg_humidity",
    "snow_depth"
]

print(df_model[cluster_features].describe())
print("\nMissing values:")
print(df_model[cluster_features].isna().sum())

In [ ]:
weather_cluster = (
    df_model[
        ["date", "station_id"] + cluster_features
    ]
    .drop_duplicates(
        subset=["date", "station_id"]
    )
    .sort_values(["date", "station_id"])
    .reset_index(drop=True)
)

print("Clustering 데이터:", weather_cluster.shape)
print("관측소 수:", weather_cluster["station_id"].nunique())
print("날짜 수:", weather_cluster["date"].nunique())

In [ ]:
train_weather = weather_cluster[
    weather_cluster["date"] <= "2021-12-31"
].copy()

val_weather = weather_cluster[
    (weather_cluster["date"] >= "2022-01-01") &
    (weather_cluster["date"] <= "2022-12-31")
].copy()

test_weather = weather_cluster[
    weather_cluster["date"] >= "2023-01-01"
].copy()

print("Train:", len(train_weather))
print("Validation:", len(val_weather))
print("Test:", len(test_weather))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    train_weather[cluster_features]
)

X_val = scaler.transform(
    val_weather[cluster_features]
)

X_test = scaler.transform(
    test_weather[cluster_features]
)

print("X_train:", X_train.shape)

In [ ]:
cluster_features = [
    "avg_temp",
    "rain",
    "avg_wind",
    "avg_humidity",
    "snow_depth"
]

print(df_model[cluster_features].isna().sum())

In [ ]:
# 적설 미기록 → 0
df_model["snow_depth"] = df_model["snow_depth"].fillna(0)

print(df_model[cluster_features].isna().sum())

In [ ]:
weather_cluster = (
    df_model[
        ["date", "station_id"] + cluster_features
    ]
    .drop_duplicates(
        subset=["date", "station_id"]
    )
    .sort_values(["station_id", "date"])
    .reset_index(drop=True)
)

print("행 수:", len(weather_cluster))
print("관측소 수:", weather_cluster["station_id"].nunique())
print("결측치:")
print(weather_cluster[cluster_features].isna().sum())

In [ ]:
weather_cluster["date"] = pd.to_datetime(weather_cluster["date"])

weather_cluster = weather_cluster.sort_values(
    ["station_id", "date"]
).reset_index(drop=True)

for col in [
    "avg_temp",
    "avg_wind",
    "avg_humidity"
]:
    weather_cluster[col] = (
        weather_cluster
        .groupby("station_id")[col]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

# rain과 snow_depth는 이미 0 처리
weather_cluster["rain"] = weather_cluster["rain"].fillna(0)
weather_cluster["snow_depth"] = weather_cluster["snow_depth"].fillna(0)

print(weather_cluster[cluster_features].isna().sum())

In [ ]:
train_weather = weather_cluster[
    weather_cluster["date"] <= "2021-12-31"
].copy()

val_weather = weather_cluster[
    (weather_cluster["date"] >= "2022-01-01") &
    (weather_cluster["date"] <= "2022-12-31")
].copy()

test_weather = weather_cluster[
    weather_cluster["date"] >= "2023-01-01"
].copy()

print("Train:", len(train_weather))
print("Validation:", len(val_weather))
print("Test:", len(test_weather))

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    train_weather[cluster_features]
)

X_val = scaler.transform(
    val_weather[cluster_features]
)

X_test = scaler.transform(
    test_weather[cluster_features]
)

print("NaN in X_train:", pd.isna(X_train).sum())

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

results = []

for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_train)

    silhouette = silhouette_score(
        X_train,
        labels
    )

    db = davies_bouldin_score(
        X_train,
        labels
    )

    ch = calinski_harabasz_score(
        X_train,
        labels
    )

    results.append({
        "K": k,
        "Silhouette": silhouette,
        "Davies_Bouldin": db,
        "Calinski_Harabasz": ch,
        "Inertia": model.inertia_
    })

cluster_scores = pd.DataFrame(results)

display(cluster_scores)

In [ ]:
from sklearn.cluster import KMeans

best_k = 6

final_kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

# Train에서만 학습
train_weather["climate_cluster"] = final_kmeans.fit_predict(X_train)

# Validation / Test는 학습된 centroid로 배정
val_weather["climate_cluster"] = final_kmeans.predict(X_val)
test_weather["climate_cluster"] = final_kmeans.predict(X_test)

weather_cluster_final = pd.concat(
    [train_weather, val_weather, test_weather],
    ignore_index=True
)

print(
    weather_cluster_final["climate_cluster"]
    .value_counts()
    .sort_index()
)

In [ ]:
cluster_profile = (
    weather_cluster_final
    .groupby("climate_cluster")[cluster_features]
    .agg(["mean", "median", "std"])
)

display(cluster_profile)

In [ ]:
cluster_mean = (
    weather_cluster_final
    .groupby("climate_cluster")[cluster_features]
    .mean()
    .round(2)
)

cluster_size = (
    weather_cluster_final
    .groupby("climate_cluster")
    .size()
    .rename("N")
)

cluster_summary = cluster_mean.join(cluster_size)

display(cluster_summary)

In [ ]:
cluster_mean = (
    weather_cluster_final
    .groupby("climate_cluster")[cluster_features]
    .mean()
    .round(2)
)

cluster_size = (
    weather_cluster_final
    .groupby("climate_cluster")
    .size()
    .rename("N")
)

cluster_summary = cluster_mean.join(cluster_size)

display(cluster_summary)

##Climate 0 — Snowy-Cold → Climate 1 — Cold-Humid → Climate 2 — Windy-Cold → Climate 3 — Cool-Dry → Climate 4 — Rainy-Humid → Climate 5 — Warm-Humid

In [ ]:
# 평균기온이 낮은 순서로 cluster 번호 재배정

temp_order = (
    weather_cluster_final
    .groupby("climate_cluster")["avg_temp"]
    .mean()
    .sort_values()
    .index
    .tolist()
)

cluster_remap = {
    old: new
    for new, old in enumerate(temp_order)
}

print("재배정:", cluster_remap)

weather_cluster_final["climate_cluster_ordered"] = (
    weather_cluster_final["climate_cluster"]
    .map(cluster_remap)
)

In [ ]:
cluster_names = {
    0: "Snowy-Cold",
    1: "Cold-Humid",
    2: "Windy-Cold",
    3: "Cool-Dry",
    4: "Rainy-Humid",
    5: "Warm-Humid"
}

weather_cluster_final["climate_regime"] = (
    weather_cluster_final["climate_cluster_ordered"]
    .map(cluster_names)
)

In [ ]:
final_cluster_summary = (
    weather_cluster_final
    .groupby(
        ["climate_cluster_ordered", "climate_regime"]
    )[cluster_features]
    .mean()
    .round(2)
)

final_cluster_summary["N"] = (
    weather_cluster_final
    .groupby(
        ["climate_cluster_ordered", "climate_regime"]
    )
    .size()
)

display(final_cluster_summary)

In [ ]:
# 날짜에서 월 추출
weather_cluster_final["date"] = pd.to_datetime(
    weather_cluster_final["date"]
)

weather_cluster_final["month"] = (
    weather_cluster_final["date"].dt.month
)

# 월별 × climate regime 건수
monthly_count = pd.crosstab(
    weather_cluster_final["month"],
    weather_cluster_final["climate_regime"]
)

# 월별 비율(%)
monthly_pct = pd.crosstab(
    weather_cluster_final["month"],
    weather_cluster_final["climate_regime"],
    normalize="index"
) * 100

monthly_pct = monthly_pct.round(1)

print("=== Monthly counts ===")
display(monthly_count)

print("\n=== Monthly distribution (%) ===")
display(monthly_pct)

In [ ]:
import matplotlib.pyplot as plt

# 원하는 순서
regime_order = [
    "Snowy-Cold",
    "Cold-Humid",
    "Windy-Cold",
    "Cool-Dry",
    "Rainy-Humid",
    "Warm-Humid"
]

monthly_plot = monthly_pct.reindex(
    columns=regime_order,
    fill_value=0
)

ax = monthly_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

ax.set_xlabel("Month")
ax.set_ylabel("Proportion (%)")
ax.set_title(
    "Monthly Distribution of Data-Driven Climate Regimes"
)

ax.set_xticklabels(
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
    rotation=0
)

ax.legend(
    title="Climate Regime",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
final_path = "/content/Gangwon_Tourism_Weather_Climate_2020_202306.csv"

df_model.to_csv(
    final_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", final_path)